# ShiftLog-Gym Notebook 3: Evaluation + Publish

Consumes Notebook 2 artifacts, builds final plots/tables, prompts for HF token, and optionally uploads.


In [ ]:
import os
REPO_URL='https://github.com/Chirag0096/ShiftLog-Gym.git'
REPO_DIR='ShiftLog-Gym'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}
!pip -q install -e . pandas matplotlib seaborn huggingface_hub


In [ ]:
import json
from getpass import getpass
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from huggingface_hub import login, create_repo, upload_folder
sns.set_theme(style='whitegrid')
OBS_ROOT=Path('observatory')
RUNS_DIR=OBS_ROOT/'training_runs'
OUTPUT_DIR=Path('artifacts/eval_publish')
PLOTS_DIR=OUTPUT_DIR/'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
HF_MODEL_REPO=os.environ.get('HF_MODEL_REPO','Chirag0096/shiftlog-gym-qwen2.5-3b-memory-policy')
PUBLISH_TO_HF=False
hf_token=os.environ.get('HF_TOKEN','').strip()
if not hf_token:
    hf_token=getpass('Enter HF_TOKEN (blank to skip upload auth): ').strip()
if hf_token:
    os.environ['HF_TOKEN']=hf_token
    login(token=hf_token)
    print('HF login successful')
else:
    print('HF login skipped')


In [ ]:
required={'stageA':RUNS_DIR/'training_curves_stageA.csv','stageB':RUNS_DIR/'training_curves_stageB.csv','stageC':RUNS_DIR/'training_curves_stageC.csv'}
missing=[k for k,v in required.items() if not v.exists()]
if missing:
    print('Missing files from notebook 2:', missing)
else:
    print('All required stage curve CSV files found')
frames={}
for k,v in required.items():
    if v.exists():
        frames[k]=pd.read_csv(v)
        print('\n',k,'tail:')
        display(frames[k].tail())


In [ ]:
def plot_stage(df, stage):
    fig,ax=plt.subplots(figsize=(10,4))
    for metric in ['reward_total','reward_recall','reward_success','reward_memory_write','recall_before_action_rate']:
        if metric in df.columns:
            ax.plot(df['step'], df[metric], label=metric)
    ax.set_title(f'{stage.upper()} training curves')
    ax.set_xlabel('step'); ax.set_ylabel('score'); ax.legend(loc='best')
    fig.tight_layout()
    path=PLOTS_DIR/f'{stage}_curves.png'
    fig.savefig(path, dpi=180)
    plt.show()
    return path
plot_paths=[]
for stage,df in frames.items():
    plot_paths.append(plot_stage(df, stage))
print('Saved plots:')
for p in plot_paths: print('-', p)


In [ ]:
summary=[]
for stage,df in frames.items():
    if df.empty: continue
    summary.append({'stage':stage,'last_reward_total':float(df['reward_total'].iloc[-1]),'last_reward_recall':float(df['reward_recall'].iloc[-1]),'last_reward_success':float(df['reward_success'].iloc[-1]),'last_recall_before_action_rate':float(df['recall_before_action_rate'].iloc[-1]) if 'recall_before_action_rate' in df.columns else 0.0})
summary_df=pd.DataFrame(summary)
summary_df.to_csv(OUTPUT_DIR/'summary_metrics.csv', index=False)
display(summary_df)
baselines_path=OBS_ROOT/'baselines.json'
if baselines_path.exists():
    baselines=json.loads(baselines_path.read_text(encoding='utf-8'))
else:
    baselines={'random':{},'scripted':{},'llm_base':{},'trained_llm':{}}
comparison=pd.DataFrame([{'agent':'Random Agent', **baselines.get('random',{})},{'agent':'Scripted Agent', **baselines.get('scripted',{})},{'agent':'Untrained LLM', **baselines.get('llm_base',{})},{'agent':'Trained LLM', **baselines.get('trained_llm',{})}])
comparison.to_csv(OUTPUT_DIR/'before_after_comparison.csv', index=False)
display(comparison)


In [ ]:
model_card='''# ShiftLog-Gym Evaluation Artifacts
This model repository stores Stage A/B/C curves, summary metrics, and baseline comparison outputs.'''
(OUTPUT_DIR/'README.md').write_text(model_card, encoding='utf-8')
print('Output bundle ready at', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file(): print('-', p)


## Optional upload to Hugging Face


In [ ]:
if PUBLISH_TO_HF:
    if not os.environ.get('HF_TOKEN'):
        raise ValueError('HF_TOKEN missing. Re-run token cell.')
    create_repo(HF_MODEL_REPO, exist_ok=True, repo_type='model')
    upload_folder(repo_id=HF_MODEL_REPO, repo_type='model', folder_path=str(OUTPUT_DIR))
    print('Uploaded to', HF_MODEL_REPO)
else:
    print('Upload skipped. Set PUBLISH_TO_HF=True when ready.')
